# 1. ICA per participant

Follows `ana01_ICA.m` of the reference pipeline (Stropahl et al., 2018; `notes/papers/eeg_data_sheet.PDF`), adapted to the RiseTogether BIDS dataset (read with `rt_bids`).

**Run once per participant.** Then run `02_corrmap.ipynb` (all participants together), then `03_clean.ipynb` (per participant).

**Flow:** 1. load (mastoids removed) → 2. events → 3. mark breaks + crop → 4. bad channels (interpolated) → 5. save the prepared data → 6. ICA training copy (low-pass 40 Hz → 250 Hz → high-pass 1 Hz) → 7. 1 s dummy epochs, reject unusual ones (joint probability > 3 SD) → 8. extended Infomax ICA → 9. save ICA and component plots

All settings are in `config.py`. Open questions about the adaptation: `ica_reference_questions.md`.

In [ ]:
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mne.preprocessing import ICA

import config as cfg  # settings; also puts rt_bids on the path
import rt_bids
from utils import find_bad_channels, firws_filter, jointprob_keep
%matplotlib inline

## 0. Participant

In [ ]:
import os

# execute_notebooks.py sets RT_SUBJECT
SUBJECT = os.environ.get("RT_SUBJECT", "sub-001")
OUT_DIR = cfg.out_dir(SUBJECT)
print(SUBJECT, f"({cfg.session_of(SUBJECT)})", "->", OUT_DIR)

## 1. Loading the raw data

One EEG recording per participant from the BIDS dataset (`sub-NNN/eeg/*.bdf`), read with `rt_bids.read_recording`. The channels already have electrode names and types (EEG, EOG2–4 as EOG), and channels marked `bad` in `channels.tsv` (at the ADC limit more than half of the time) are already in `raw.info["bads"]`. The mastoids (M1, M2) are removed.

In [ ]:
raw = rt_bids.read_recording(cfg.BIDS_ROOT, SUBJECT).load_data()
print(f"{SUBJECT}: {raw.info['nchan']} channels, {raw.times[-1]:.1f} s, {raw.info['sfreq']} Hz")
print("Marked bad in the dataset:", raw.info["bads"])

raw.drop_channels(cfg.DROP_CHANNELS)
raw.set_channel_types({ch: "eog" for ch in cfg.EOG_CHANNELS})
raw.set_montage(cfg.MONTAGE)

# lets rt_bids find this recording's session clock later (kept in the saved .fif)
raw.info["description"] = "rt_bids:" + str(cfg.bdf_path(SUBJECT).resolve())
print(f"EEG channels: {len(mne.pick_types(raw.info, eeg=True))}, EOG channels: {len(mne.pick_types(raw.info, eog=True))}")

## 2. Events

Every event of `events.tsv` is an annotation at its true sample in this recording (`rt_bids` leaves out the sync-only events). Skipped samples (`acq_gap`) are only counted (saved in `_ica_info.csv`).

In [ ]:
assert len(raw.annotations) > 0, f"{SUBJECT}: no annotations"

descriptions = pd.Series(raw.annotations.description)
n_acq_gaps = int(descriptions.str.startswith("acq_gap").sum())
print(f"{len(raw.annotations)} annotations; skipped-sample gaps (acq_gap): {n_acq_gaps}")

trial_starts = descriptions[descriptions.str.startswith(cfg.TRIAL_START)]
print(trial_starts.str.split("/").str[1].value_counts())

events, event_id = mne.events_from_annotations(raw, regexp="^TRIAL_(START|END)/")
fig = mne.viz.plot_events(events, event_id=event_id, sfreq=raw.info["sfreq"], first_samp=raw.first_samp)

### Check: are all files of the session complete?

Every participant should have the same number of trials per condition. Start times are not compared: the headsets were switched on one at a time.

In [ ]:
rows = []
for sub, r in rt_bids.read_session(cfg.BIDS_ROOT, cfg.session_of(SUBJECT)).items():
    starts = [d.split("/")[1] for d in r.annotations.description if d.startswith(cfg.TRIAL_START)]
    row = {"subject": sub, "sfreq": r.info["sfreq"], "n_channels": r.info["nchan"],
           "bads_in_dataset": " ".join(r.info["bads"])}
    row.update(pd.Series(starts, dtype=str).value_counts().to_dict())
    rows.append(row)
pd.DataFrame(rows)

## 3. Marking breaks and cropping to the task

The recording covers the whole session. Long gaps between trials (surveys, results screens, instructions, team reassignment) are marked as `BAD_break`, so ICA does not learn from them; then the data is cropped to the trials. The motor baseline is kept (`cfg.CONDITIONS`).

Breaks are marked **before** cropping, while the data still start at sample 0, so their onsets need no correction.

In [ ]:
def trial_times(raw, prefix):
    """Times (s, from the start of the data) of the TRIAL_START/ or TRIAL_END/ events of the kept conditions."""
    pattern = "^" + prefix + "(" + "|".join(cfg.CONDITIONS) + ")/"
    ev, _ = mne.events_from_annotations(raw, regexp=pattern, verbose="error")
    return np.sort((ev[:, 0] - raw.first_samp) / raw.info["sfreq"])


starts, ends = trial_times(raw, cfg.TRIAL_START), trial_times(raw, cfg.TRIAL_END)

break_onsets, break_durations = [], []
for end, next_start in zip(ends[:-1], starts[1:]):
    if next_start - end > cfg.BREAK_MIN_DURATION:
        break_onsets.append(end + 1.0)
        break_durations.append(next_start - end - 2.0)  # 1 s margin on each side

onset_offset = raw.first_time if raw.annotations.orig_time is not None else 0.0
breaks = mne.Annotations(onset=np.array(break_onsets) + onset_offset, duration=break_durations,
                         description="BAD_break", orig_time=raw.annotations.orig_time)
raw.set_annotations(raw.annotations + breaks)

raw.crop(tmin=max(starts.min() - 2.0, 0), tmax=min(ends.max() + 2.0, raw.times[-1]))
print(f"Breaks marked: {len(breaks)}, total {np.sum(break_durations):.1f} s of {raw.times[-1]:.1f} s")

## 4. Bad channels

Bad channels come from three sources: channels marked bad in the dataset, **clearly bad channels found automatically** (`utils.find_bad_channels`: noisy, flat or uncorrelated with every other channel, criteria of the PREP pipeline; thresholds in `config.py`), and any you add by hand to `BAD_CHANNELS` in `config.py` after looking at the data. The printed median channel SD is also a units check: it should be tens of µV (question 18).

Bad channels are **interpolated before ICA**: CORRMAP compares component maps across participants, so every participant needs the same channels. (Tip: `%matplotlib qt` gives interactive plots.)

In [ ]:
# units check: should be tens of µV
sd = raw.copy().filter(1, 40, picks="eeg", verbose="error").get_data(picks="eeg", reject_by_annotation="omit").std(axis=1)
print(f"Median channel SD (1-40 Hz): {np.median(sd) * 1e6:.1f} µV")

raw.plot(n_channels=30, scalings=dict(eeg=50e-6))
raw.compute_psd(fmax=60).plot()

In [ ]:
import importlib
importlib.reload(cfg)  # pick up changes made to config.py

dataset_bads = list(raw.info["bads"])
channel_check = find_bad_channels(raw, cfg.BAD_CHANNEL_Z, cfg.BAD_CHANNEL_CORR, cfg.BAD_CHANNEL_CORR_FRACTION,
                                  cfg.BAD_CHANNEL_FLAT_UV)
auto_bads = channel_check.loc[channel_check["reason"] != "", "channel"].tolist()
manual_bads = cfg.BAD_CHANNELS.get(SUBJECT, [])
bad_channels = sorted(set(dataset_bads) | set(auto_bads) | set(manual_bads))
raw.info["bads"] = bad_channels
print("Bad channels:", bad_channels, "| dataset:", dataset_bads, "| automatic:", auto_bads, "| by hand:", manual_bads)
print(channel_check.sort_values("sd_z", ascending=False).to_string(index=False))
if bad_channels:
    raw.interpolate_bads(reset_bads=True)

## 5. Saving the prepared data

The cropped data at the original sampling rate (500 Hz), with breaks marked and bad channels interpolated. `03_clean.ipynb` applies the ICA to this file (as `ana01` writes the ICA weights to the original data).

In [ ]:
raw.save(OUT_DIR / f"{SUBJECT}_prep_raw.fif", overwrite=True)

## 6. ICA training copy

Exactly the order of `ana01_ICA.m`: low-pass 40 Hz (order 100) → downsample to 250 Hz → high-pass 1 Hz (order 500). Filters are windowed-sinc FIR with a Hann window, like EEGLAB's `pop_firws`. This copy is only used to learn the ICA.

In [ ]:
raw_ica = raw.copy()
firws_filter(raw_ica, cfg.ICA_LP, "lowpass", cfg.ICA_LP_ORDER)
raw_ica.resample(cfg.ICA_SRATE)
firws_filter(raw_ica, cfg.ICA_HP, "highpass", cfg.ICA_HP_ORDER)

## 7. Dummy epochs and joint-probability rejection

The continuous copy is cut into 1 s pieces ("dummy epochs", `eeg_regepochs`), skipping the breaks. Pieces with very unusual values (joint probability more than 3 SD from the mean, per channel or over all channels, as `pop_jointprob`) are removed, so large artifacts do not dominate the ICA.

In [ ]:
epochs = mne.make_fixed_length_epochs(raw_ica, duration=cfg.ICA_EPOCH_LENGTH, reject_by_annotation=True,
                                      preload=True)
keep = jointprob_keep(epochs.get_data(picks="eeg"), cfg.ICA_PRUNE)
epochs_ica = epochs[np.where(keep)[0]]
print(f"Dummy epochs: {len(epochs)}, rejected {np.sum(~keep)} ({100 * np.mean(~keep):.1f}%), kept {len(epochs_ica)}")

## 8. ICA

Extended Infomax (EEGLAB's `runica` with `'extended', 1`), on the EEG channels. The PCA option of the reference keeps 50 dimensions; it is capped at the rank of the data (the number of EEG channels minus the interpolated ones), because there are fewer than 50 channels.

In [ ]:
rank = len(mne.pick_types(raw_ica.info, eeg=True)) - len(bad_channels)  # interpolated channels add no information
n_components = min(cfg.ICA_PCADIMS, rank) if cfg.ICA_PCA else rank
print(f"Data rank: {rank}, ICA components: {n_components}")

ica = ICA(
    n_components=n_components,
    method="infomax",
    fit_params=dict(extended=True),  # extended Infomax = EEGLAB runica 'extended', 1
    max_iter="auto",
    random_state=cfg.RANDOM_STATE,
)
ica.fit(epochs_ica, picks="eeg")
print(ica)

## 9. Saving the ICA and the component plots

The ICA solution is saved for `02_corrmap.ipynb`. The component topographies are saved as PNG (as in `ana02`, part 1): use them to choose the CORRMAP templates.

In [ ]:
ica.save(OUT_DIR / f"{SUBJECT}_ica.fif", overwrite=True)

figures = ica.plot_components(show=False)
figures = figures if isinstance(figures, list) else [figures]
for i, figure in enumerate(figures):
    figure.suptitle(SUBJECT)
    figure.savefig(OUT_DIR / f"{SUBJECT}_ica_components_{i + 1}.png", dpi=100)
    plt.close(figure)

pd.DataFrame([{
    "subject": SUBJECT,
    "bad_channels": " ".join(bad_channels),
    "n_bad_channels": len(bad_channels),
    "bad_channels_dataset": " ".join(dataset_bads),
    "bad_channels_auto": " ".join(auto_bads),
    "break_s": round(float(np.sum(break_durations)), 1),
    "n_acq_gaps": n_acq_gaps,
    "n_dummy_epochs": len(epochs),
    "n_rejected_epochs": int(np.sum(~keep)),
    "n_ica_components": ica.n_components_,
}]).to_csv(OUT_DIR / f"{SUBJECT}_ica_info.csv", index=False)
print("Saved to", OUT_DIR)